# Pre-processing and evaluation of meteorological reanalyses

Code accompanying *Boisvert-Vigneault et al. (2026), "Simulating locked pastures in snow from Arctic rain events"*.

This notebook
1. partitions reanalysis precipitation into rain and snow with four precipitation phase partitioning methods (PPMs: J18, HP13, W19, B00) and stores the result in one netCDF file (**Section 1**),
2. evaluates ERA5 and CaSRv3.2 against the Banks Island weather stations (LWS): temperature, wind, humidity and daily precipitation occurrence (**Sections 3-4**, Tables 3, 4 and C1),
3. evaluates the PPMs against observed precipitation phase at Sachs Harbour (**Section 5**, Fig. 5 data and Table 5, plus the freezing-rain statistics of Section 3.2),
4. creates the forcing files of the *cold* and *no-rain* sensitivity simulations (**Section 6**, Section 2.5),
5. summarises the SVS2-Crocus ensemble outputs (**Section 7**, Section 3.3.1 and Fig. 8).

`Final_figures.ipynb` produces the manuscript figures and reads the files written here.

**Requirements:** numpy, pandas (>= 2.0), xarray, netCDF4, scipy, metpy.

**Folder layout** (everything is relative to `PROJECT_DIR`, set with the environment variable `ROS_PROJECT_DIR`, default: current directory):

```
PROJECT_DIR/
  Data/RAW/Banks_weather_stations/   *_8020.csv (hourly LWS) and 2503650_AdjDly.csv (adjusted daily precipitation)
  Data/RAW/met_extractions/          <CaSR32_analysis | CaSR32_prediction | ERA5>/basin_forcing_<STATION>_8010_<casr32|era5>_forcing.met
                                     and, for B00, extra_pcpn_casr3_<STATION>_8010_casr32_forcing.met in the CaSR32_* folders
  Data/PROCESSED/PPM/                written here (one sub-folder per PPM, plus the netCDF and Manual/)
  Results/                           written here (score tables); SVS2-Crocus ensemble summaries are read from here
```

# Setup

In [ ]:
import os
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from scipy import stats
from metpy.calc import (dewpoint_from_specific_humidity,
                        relative_humidity_from_specific_humidity,
                        wet_bulb_temperature)
from metpy.units import units

## Paths and parameters

In [ ]:
PROJECT_DIR = Path(os.environ.get('ROS_PROJECT_DIR', Path.cwd()))

LWS_DIR = PROJECT_DIR / 'Data' / 'RAW' / 'Banks_weather_stations'
RAW_MET_DIR = PROJECT_DIR / 'Data' / 'RAW' / 'met_extractions'
PPM_DIR = PROJECT_DIR / 'Data' / 'PROCESSED' / 'PPM'
SCENARIO_DIR = PPM_DIR / 'Manual'          # forcing of the cold / no-rain simulations
PPM_NETCDF = PPM_DIR / 'basin_forcing_PPMs_1980-2020_elevcorrF.nc'
RESULTS_DIR = PROJECT_DIR / 'Results'

STATIONS = ['SACHS_A', 'THOMSEN_RIVER', 'NANGMAGVIK_LAKE']
REANALYSES = ['CaSR32_analysis', 'CaSR32_prediction', 'ERA5']
PPMS = ['HP13', 'W19', 'J18', 'B00']

MET_START = pd.Timestamp('1980-08-30 13:00')   # common hourly time grid of all forcing files
MET_END = pd.Timestamp('2020-08-01 00:00')

# Rain/snow air-temperature threshold per station (deg C), from Jennings et al. (2018)
J18_THRESHOLDS = {'SACHS_A': 1.22145, 'THOMSEN_RIVER': 1.34865, 'NANGMAGVIK_LAKE': 1.33991}

NEAR_FREEZING_RANGE = (-5, 5)   # deg C, LWS air temperature defining "near-freezing" periods
MAX_TEMP_DIFF = 5               # deg C, max |LWS - reanalysis| temperature difference tolerated in near-freezing periods
TRACE_PRECIP_MM = 0.2           # mm/day, minimum precipitation defining a precipitation day (hourly: / 24)

REBUILD_PPM_FILES = False       # True: recompute the PPM forcing files and netCDF even if the netCDF exists

for folder in [RESULTS_DIR, SCENARIO_DIR] + [PPM_DIR / ppm for ppm in PPMS]:
    folder.mkdir(parents=True, exist_ok=True)

# 1. Precipitation phase partitioning (PPMs) and netCDF

## Functions

In [ ]:
# ----- Forcing-file (.met) input/output ------------------------------------------------

FORCING_COLUMNS = ['HOUR', 'MINS', 'JDAY', 'YEAR', 'FSIN', 'FLIN', 'PRE',
                   'TA', 'QA', 'UV', 'PRES', 'PRERN', 'PRESNO']

VAR_ATTRS = {
    'FSIN':   ('Incoming shortwave radiation at the surface',       'W m**-2'),
    'FLIN':   ('Incoming longwave radiation at the surface',        'W m**-2'),
    'PRE':    ('Total precipitation rate at the surface',           'kg m**-2 s**-1'),
    'TA':     ('Air temperature at the forcing reference height',   'degrees C'),
    'QA':     ('Specific humidity at the forcing reference height', 'kg kg**-1'),
    'UV':     ('Wind speed at the forcing reference height',        'm s**-1'),
    'PRES':   ('Surface pressure',                                  'Pa'),
    'PRERN':  ('Total incoming liquid precipitation rate',          'kg m**-2 s**-1'),
    'PRESNO': ('Total incoming solid precipitation rate',           'kg m**-2 s**-1'),
}

PPM_ATTRS = {
    'HP13': 'Harder & Pomeroy, 2013: Estimating Precipitation Phase Using a Psychrometric Energy Balance Method',
    'W19':  'Wang et al., 2019: A Wet-Bulb Temperature-Based Rain-Snow Partitioning Scheme Improves Snowpack Prediction Over the Drier Western United States',
    'J18':  'Jennings et al., 2018: Spatial variation of the rain-snow temperature threshold across the Northern Hemisphere',
    'B00':  'Bourgouin 2000: A Method to Determine Precipitation Types',
}


def met_timestamps(df):
    """Build timestamps from the YEAR, JDAY, HOUR and MINS columns of a .met table."""
    return (pd.to_datetime(df['YEAR'], format='%Y')
            + pd.to_timedelta(df['JDAY'] - 1, unit='D')
            + pd.to_timedelta(df['HOUR'], unit='h')
            + pd.to_timedelta(df['MINS'], unit='m'))


def read_basin_forcing(path):
    """Read a raw basin_forcing .met file (11 or 13 columns) into a time-indexed DataFrame."""
    with open(path) as f:
        n_cols = len(f.readline().split())
    df = pd.read_csv(path, sep=r'\s+', header=None, names=FORCING_COLUMNS[:n_cols])
    return df.set_index(met_timestamps(df))


def read_b00_precip(path):
    """Read the four-phase precipitation rates (kg m-2 s-1) produced by B00 (extra_pcpn_* file)."""
    df = pd.read_csv(path, sep=',', header=0,
                     names=['HOUR', 'MINS', 'JDAY', 'YEAR', 'Rain', 'Snow', 'Ice Pellets', 'Freezing Rain'])
    return df.set_index(met_timestamps(df))[['Rain', 'Snow', 'Ice Pellets', 'Freezing Rain']]


def read_forcing_on_grid(path, time_index):
    """Read a .met file onto a fixed hourly grid; returns all-NaN data if the file is missing."""
    path = Path(path)
    if not path.exists():
        print(f'    File not found: {path}')
        return pd.DataFrame(np.nan, index=time_index, columns=list(VAR_ATTRS))

    df = pd.read_csv(path, sep=r'\s+', header=None, names=FORCING_COLUMNS).drop_duplicates()
    df.index = met_timestamps(df)

    n_dupes = df.index.duplicated().sum()
    if n_dupes:
        print(f'    Warning: {n_dupes} duplicate timestamps in {path.name}, keeping the first occurrence')
        df = df[~df.index.duplicated(keep='first')]

    n_off_grid = len(df.index.difference(time_index))
    if n_off_grid:
        print(f'    Warning: {n_off_grid} timestamps in {path.name} are outside the target time grid and are dropped')

    return df[list(VAR_ATTRS)].reindex(time_index)


def write_met_file(df, path):
    """Write a forcing table (columns FORCING_COLUMNS) as a space-separated .met file without header."""
    df[FORCING_COLUMNS].to_csv(path, header=False, index=False, sep=' ', decimal='.')


def trim_empty_edges(path):
    """Remove leading/trailing rows without data (only HOUR/MINS/JDAY/YEAR) from an existing .met file.
    Returns True if the file was rewritten."""
    with open(path) as f:
        lines = f.readlines()

    has_data = [len(line.split()) > 4 for line in lines]
    if not any(has_data):
        print(f'    WARNING: {path} has no valid data - left untouched')
        return False
    first = has_data.index(True)
    last = len(lines) - 1 - has_data[::-1].index(True)
    if first == 0 and last == len(lines) - 1:
        return False

    with open(path, 'w') as f:
        f.writelines(lines[first:last + 1])
    print(f'    Cleaned {Path(path).name}: removed {first} leading and {len(lines) - 1 - last} trailing empty rows')
    return True

In [ ]:
# ----- Precipitation phase partitioning methods: rain fraction (0 = snow, 1 = rain) ----

def _hp13_residual(ti, ta, rh):
    """Residual f(Ti) of the psychrometric energy balance of Harder and Pomeroy (2013) and its derivative."""
    tk = ta + 273.15
    mw = 0.01801528 * 1000.0                   # molar mass of water
    r = 8.31441 / 1000.0                       # gas constant
    d = 2.06e-5 * (tk / 273.15) ** 1.75        # water-vapour diffusivity
    lambda_t = 0.000063 * tk + 0.00673         # thermal conductivity of air
    latent = np.where(ta < 0,
                      1000.0 * (2834.1 - 0.29 * ta - 0.004 * ta ** 2),   # sublimation
                      1000.0 * (2501.0 - 2.361 * ta))                    # vaporisation
    ea = rh / 100 * 0.611 * np.exp((17.3 * ta) / (237.3 + ta))
    rho = (mw * ea) / (r * tk)
    exp_ti = np.exp(17.3 * ti / (237.3 + ti))
    tk_i = ti + 273.15
    f = ta + d * latent * (rho / 1000.0 - 0.611 * mw * exp_ti / (r * tk_i * 1000.0)) / lambda_t - ti
    df = (d * latent * (-0.611e-3 * mw * (17.3 / (237.3 + ti) - 17.3 * ti / (237.3 + ti) ** 2)
                        * exp_ti / (r * tk_i)
                        + 0.611e-3 * mw * exp_ti / (r * tk_i ** 2)) / lambda_t - 1)
    return f, df


def rain_fraction_hp13(ta, qa, pres):
    """Harder and Pomeroy (2013): hydrometeor temperature from a psychrometric energy balance.
    ta in deg C, qa in kg/kg, pres in Pa."""
    b, c = 2.50286, 0.125006
    ta, qa, pres = np.asarray(ta), np.asarray(qa), np.asarray(pres)
    rh = relative_humidity_from_specific_humidity(
        (pres / 100) * units.hPa, ta * units.degC, qa * units('kg/kg')).to('percent').magnitude

    ti = ta.copy()                              # hydrometeor temperature, vectorised Newton-Raphson
    for _ in range(50):
        f, df = _hp13_residual(ti, ta, rh)
        ti_new = ti - f / df
        if np.max(np.abs(ti_new - ti)) <= 1e-2:
            break
        ti = ti_new
    return 1.0 / (1 + b * c ** ti)


def rain_fraction_w19(ta, qa, pres):
    """Wang et al. (2019): wet-bulb temperature based rain fraction."""
    a, b, c = 6.99e-5, 2, 3.97
    ta, qa, pres = (np.asarray(x, dtype=np.float64) for x in (ta, qa, pres))
    pres_hpa = (pres / 100) * units.hPa
    td = dewpoint_from_specific_humidity(pres_hpa, ta * units.degC, qa * units('kg/kg')).to('degC')
    tw = wet_bulb_temperature(pres_hpa, ta * units.degC, td).to('degC').magnitude
    return 1.0 - 1.0 / (1.0 + a * np.exp(b * (tw + c)))


def rain_fraction_j18(ta, threshold):
    """Jennings et al. (2018): rain above a fixed air-temperature threshold, snow below it."""
    ta = np.asarray(ta)
    return np.where(ta > threshold, 1.0, np.where(ta < threshold, 0.0, 0.5))


def to_crocus_forcing(df, rain_fraction):
    """Split total precipitation (PRE) with `rain_fraction` and return the table in .met format."""
    out = df.copy()
    out['PRERN'] = out['PRE'] * rain_fraction
    out['PRESNO'] = out['PRE'] * (1 - rain_fraction)
    out['HOUR'], out['MINS'] = out.index.hour, out.index.minute
    out['JDAY'], out['YEAR'] = out.index.dayofyear, out.index.year
    return out[FORCING_COLUMNS]


def build_b00_forcing(product, station):
    """CaSRv3.2 forcing with the rain/snow split of Bourgouin (2000).
    Rain + freezing rain count as liquid, snow + ice pellets as solid."""
    raw_dir = RAW_MET_DIR / f'CaSR32_{product}'
    forcing = read_basin_forcing(raw_dir / f'basin_forcing_{station}_8010_casr32_forcing.met')
    phases = read_b00_precip(raw_dir / f'extra_pcpn_casr3_{station}_8010_casr32_forcing.met')
    assert len(forcing) == len(phases), f'Forcing and B00 files differ in length ({product}, {station})'
    forcing['PRERN'] = (phases['Rain'] + phases['Freezing Rain']).to_numpy()
    forcing['PRESNO'] = (phases['Snow'] + phases['Ice Pellets']).to_numpy()
    return forcing

In [ ]:
# ----- netCDF assembly -----------------------------------------------------------------

def build_forcing_dataset(base_dir, stations, reanalyses, start, end, ppms=None):
    """Assemble forcing files into one xarray Dataset on a fixed hourly grid.

    ppms=None : reads <base_dir>/<reanalysis>/basin_forcing_<station>_8010_<casr32|era5>_forcing.met
    ppms=list : reads <base_dir>/<ppm>/basin_forcing_<station>_<reanalysis>.met and adds a 'PPM' dimension.
                Every variable carries the PPM dimension, because the B00 files come from a separate
                extraction and are not guaranteed to be identical to the others for non-precipitation variables.
    """
    time_index = pd.date_range(start=start, end=end, freq='h')
    dims = ['forecast', 'location'] + (['PPM'] if ppms else []) + ['time']
    shape = (len(reanalyses), len(stations)) + ((len(ppms),) if ppms else ()) + (len(time_index),)
    arrays = {var: np.full(shape, np.nan) for var in VAR_ATTRS}

    for j, reanalysis in enumerate(reanalyses):
        for k, station in enumerate(stations):
            print(f'    {reanalysis} - {station}')
            if ppms:
                for m, ppm in enumerate(ppms):
                    if (reanalysis, ppm) == ('ERA5', 'B00'):     # B00 only exists for CaSRv3.2
                        continue
                    df = read_forcing_on_grid(base_dir / ppm / f'basin_forcing_{station}_{reanalysis}.met', time_index)
                    for var in VAR_ATTRS:
                        arrays[var][j, k, m, :] = df[var].to_numpy()
            else:
                slug = reanalysis.lower().split('_')[0]           # 'casr32' or 'era5'
                df = read_forcing_on_grid(base_dir / reanalysis / f'basin_forcing_{station}_8010_{slug}_forcing.met', time_index)
                for var in VAR_ATTRS:
                    arrays[var][j, k, :] = df[var].to_numpy()

    coords = {'time': time_index, 'forecast': reanalyses, 'location': stations}
    if ppms:
        coords['PPM'] = ppms
    ds = xr.Dataset({var: (dims, arrays[var]) for var in VAR_ATTRS}, coords=coords)
    for var, (description, var_units) in VAR_ATTRS.items():
        ds[var].attrs.update({'description': description, 'units': var_units})
    if ppms:
        ds['PPM'].attrs.update({p: PPM_ATTRS[p] for p in ppms})
    return ds

## Compute the PPMs and save the netCDF

In [ ]:
if REBUILD_PPM_FILES or not PPM_NETCDF.exists():
    print('Reading the raw forcing files...')
    base_ds = build_forcing_dataset(RAW_MET_DIR, STATIONS, REANALYSES, MET_START, MET_END)

    # Ground-based PPMs (J18, W19, HP13): computed from the reanalysis air temperature and humidity
    print('\nApplying the ground-based PPMs...')
    for station in STATIONS:
        for reanalysis in REANALYSES:
            df = base_ds.sel(location=station, forecast=reanalysis).to_dataframe()
            for ppm in ['HP13', 'J18', 'W19']:
                out_file = PPM_DIR / ppm / f'basin_forcing_{station}_{reanalysis}.met'
                if out_file.exists():
                    trim_empty_edges(out_file)
                    continue
                print(f'    {station} - {reanalysis} - {ppm}')
                if ppm == 'HP13':
                    fraction = rain_fraction_hp13(df.TA, df.QA, df.PRES)
                elif ppm == 'W19':
                    fraction = rain_fraction_w19(df.TA, df.QA, df.PRES)
                else:
                    fraction = rain_fraction_j18(df.TA, J18_THRESHOLDS[station])
                write_met_file(to_crocus_forcing(df, fraction), out_file)

    # B00: four-phase output of the CaSRv3.2 extraction, available for CaSRv3.2 only
    print('\nAdding B00...')
    for product in ['prediction', 'analysis']:
        for station in STATIONS:
            out_file = PPM_DIR / 'B00' / f'basin_forcing_{station}_CaSR32_{product}.met'
            if out_file.exists():
                trim_empty_edges(out_file)
                continue
            print(f'    {station} - CaSR32_{product}')
            write_met_file(build_b00_forcing(product, station), out_file)

    print('\nSaving the netCDF...')
    ppm_ds = build_forcing_dataset(PPM_DIR, STATIONS, REANALYSES, MET_START, MET_END, ppms=PPMS)
    ppm_ds.to_netcdf(PPM_NETCDF)
    print(f'Saved {PPM_NETCDF}')
else:
    print(f'{PPM_NETCDF.name} already exists - skipping (set REBUILD_PPM_FILES = True to recompute)')

# 2. Load the PPM dataset and the weather-station data

In [ ]:
def load_lws_stations(folder):
    """Hourly ECCC station data (*_8020.csv), as {station: DataFrame(time, weather, pressure, rel_hum, temp, wind_spd)}."""
    stations = {}
    for csv in sorted(Path(folder).glob('*_8020.csv')):
        df = pd.read_csv(csv, low_memory=False)
        station = df['station_name'][0].replace(' ', '_')
        if 'SACHS' in station:
            station = 'SACHS_A'
        df = df[['time', 'weather', 'pressure', 'rel_hum', 'temp', 'wind_spd']].copy()
        df['time'] = pd.to_datetime(df['time'], format='mixed')
        stations[station] = df
    return stations


def load_adjusted_daily_precip(path):
    """Adjusted Daily Rainfall and Snowfall Data for Canada (Wang et al., 2017), Sachs Harbour.
    Returns daily total precipitation (mm) indexed by date; missing days (negative flags) are dropped."""
    df = pd.read_csv(path, sep=';')
    total = df['cR+cS(mm)'].astype(str).str.lower().str.replace('[a-z]', '', regex=True).astype(float)   # strip quality flags
    dates = pd.date_range('1955-11-01', '2013-10-31', freq='D')
    assert len(dates) == len(df), 'The adjusted daily file no longer matches the 1955-11-01 to 2013-10-31 date range'
    series = pd.Series(total.to_numpy(), index=dates, name='Total_precip')
    return series[series >= 0]


ppm_ds = xr.open_dataset(PPM_NETCDF)
lws = load_lws_stations(LWS_DIR)
adj_daily_precip = load_adjusted_daily_precip(LWS_DIR / '2503650_AdjDly.csv')

print(f'PPM dataset dimensions: {dict(ppm_ds.sizes)}')
print(f'Weather stations: {list(lws)}')
print(f'Adjusted daily precipitation: {adj_daily_precip.index[0]:%Y-%m-%d} to {adj_daily_precip.index[-1]:%Y-%m-%d}, '
      f'{len(adj_daily_precip)} valid days')

# 3. Air temperature, wind speed and relative humidity: full dataset (Table 3, part i)

## Functions

In [ ]:
def rmse(obs, model):
    return float(np.sqrt(np.mean((np.asarray(model) - np.asarray(obs)) ** 2)))


def comparison_stats(obs, model):
    """Bias, RMSD and regression statistics of `model` against `obs`."""
    obs, model = np.asarray(obs), np.asarray(model)
    regression = stats.linregress(obs, model)
    return {'bias': float(np.mean(model - obs)), 'rmsd': rmse(obs, model),
            'r2': regression.rvalue ** 2, 'slope': regression.slope, 'p_value': float(regression.pvalue),
            'pbias': 100 * float(np.sum(model - obs) / np.sum(obs))}


def sync_series(obs_values, obs_time, model_values, model_time):
    """Keep the timestamps present in both series, within their common time window.
    Returns a DataFrame with columns time, obs, model."""
    obs = pd.DataFrame({'time': pd.to_datetime(obs_time), 'obs': np.asarray(obs_values)})
    model = pd.DataFrame({'time': pd.to_datetime(model_time), 'model': np.asarray(model_values)})
    start = max(obs['time'].min(), model['time'].min())
    end = min(obs['time'].max(), model['time'].max())
    obs = obs[(obs['time'] >= start) & (obs['time'] <= end)]
    model = model[(model['time'] >= start) & (model['time'] <= end)]
    return obs.merge(model, on='time', how='inner').sort_values('time').reset_index(drop=True)


# variable -> (LWS column, forcing variable)
VARIABLES = {'temperature': ('temp', 'TA'),
             'humidity': ('rel_hum', 'QA'),
             'wind': ('wind_spd', 'UV')}
COMPARED_REANALYSES = ['CaSR32_analysis', 'ERA5']   # the precipitation-only CaSRv3.2 forecast shares these variables


def compare_with_lws(ppm_ds, lws, variable):
    """Hourly pairs of LWS observations and reanalysis values for every reanalysis and station.

    Returns ({reanalysis: DataFrame(time, obs, model, station)}, DataFrame of statistics).
    The HP13 slice of the PPM dimension is used, since only precipitation depends on the PPM.
    Wind speed: the LWS reports km/h, converted here to m/s.
    """
    lws_column, forcing_variable = VARIABLES[variable]
    base = ppm_ds.sel(PPM='HP13')
    pairs, records = {}, []

    for reanalysis in COMPARED_REANALYSES:
        station_pairs = []
        for station in base.location.values:
            ds = base.sel(forecast=reanalysis, location=station)
            if variable == 'humidity':
                model = relative_humidity_from_specific_humidity(
                    (ds['PRES'].values / 100) * units.hPa, ds['TA'].values * units.degC,
                    ds['QA'].values * units('kg/kg')).to('percent').magnitude
            else:
                model = ds[forcing_variable].values

            station_lws = lws[station]
            df = sync_series(station_lws[lws_column].to_numpy(), station_lws['time'].to_numpy(),
                             model, ds.time.values)
            if variable == 'wind':
                df['obs'] = df['obs'] / 3.6
            df = df.dropna()
            df['station'] = station
            records.append({'station': station, 'reanalysis': reanalysis,
                            **comparison_stats(df['obs'], df['model'])})
            station_pairs.append(df)
        pairs[reanalysis] = pd.concat(station_pairs, ignore_index=True)

    return pairs, pd.DataFrame.from_records(records)

## Bias and RMSD, full dataset

In [ ]:
pairs, full_stats = {}, {}
for variable in VARIABLES:
    pairs[variable], full_stats[variable] = compare_with_lws(ppm_ds, lws, variable)

COLUMNS_SHOWN = ['station', 'reanalysis', 'bias', 'rmsd', 'r2']
for variable, table in full_stats.items():
    print(f'\n{variable.upper()} - full dataset')
    print(table[COLUMNS_SHOWN].round(2).to_string(index=False))

# 4. Daily precipitation occurrence at Sachs Harbour (Table 4)

In [ ]:
def occurrence_scores(a, b, c, d):
    """Frequency Bias Index, Equitable Threat Score and proportion correct from a 2x2 contingency table.
    a: hits, b: false alarms, c: misses, d: correct negatives (Appendix A)."""
    chance_hits = (a + b) * (a + c) / (a + b + c + d)
    return {'FBI': (a + b) / (a + c),
            'ETS': (a - chance_hits) / (a + b + c - chance_hits),
            'PC': (a + d) / (a + b + c + d)}


def daily_reanalysis_precip(ppm_ds, station='SACHS_A'):
    """Daily total precipitation (mm) per reanalysis; days below the trace threshold are set to zero."""
    daily = {}
    for reanalysis in ppm_ds.forecast.values:
        pre = ppm_ds['PRE'].sel(forecast=reanalysis, location=station, PPM='HP13')
        hourly_mm = pd.Series(pre.values * 3600, index=pd.to_datetime(pre.time.values))   # kg m-2 s-1 -> mm per hour
        series = hourly_mm.resample('1D').sum()
        series[series < TRACE_PRECIP_MM] = 0
        daily[reanalysis] = series
    return daily


daily_precip = daily_reanalysis_precip(ppm_ds)

rows = []
for reanalysis, series in daily_precip.items():
    pair = sync_series(adj_daily_precip.to_numpy(), adj_daily_precip.index, series.to_numpy(), series.index)
    obs_wet, model_wet = pair['obs'] > 0, pair['model'] > 0
    a = (obs_wet & model_wet).sum()
    b = (~obs_wet & model_wet).sum()
    c = (obs_wet & ~model_wet).sum()
    d = (~obs_wet & ~model_wet).sum()
    rows.append({'reanalysis': reanalysis, **occurrence_scores(a, b, c, d),
                 'missed_days_pct_of_all_days': 100 * c / (a + b + c + d)})

print('Daily precipitation occurrence, all temperatures (Table 4)')
print(pd.DataFrame(rows).round(3).to_string(index=False))

# 5. Near-freezing periods (LWS temperature between -5 and 5 °C)

## Select the near-freezing periods and evaluate the variables (Table 3 part ii, Table C1)

In [ ]:
def select_near_freezing(df, max_temp_diff=None):
    """Rows of an (obs, model) temperature DataFrame where the LWS temperature is in NEAR_FREEZING_RANGE
    and, optionally, the reanalysis is within `max_temp_diff` of the LWS."""
    low, high = NEAR_FREEZING_RANGE
    selected = df[(df['obs'] >= low) & (df['obs'] <= high)]
    if max_temp_diff is not None:
        selected = selected[(selected['obs'] - selected['model']).abs() <= max_temp_diff]
    return selected


near_freezing_times = {}   # (station, reanalysis or 'LWS') -> timestamps
sample_sizes = []
for reanalysis, df_all in pairs['temperature'].items():
    for station in STATIONS:
        df = df_all[df_all['station'] == station]
        in_range = select_near_freezing(df)
        kept = select_near_freezing(df, MAX_TEMP_DIFF)
        if reanalysis == 'ERA5':          # LWS-only selection (longest record), used to evaluate temperature itself
            near_freezing_times[(station, 'LWS')] = in_range['time']
        near_freezing_times[(station, reanalysis)] = kept['time']
        sample_sizes.append({'station': station, 'reanalysis': reanalysis,
                             'data_start': df['time'].iloc[0], 'data_end': df['time'].iloc[-1],
                             'n_total': len(df), 'n_near_freezing': len(in_range),
                             'near_freezing_pct': round(100 * len(in_range) / len(df), 1),
                             'n_removed_by_temp_diff': len(in_range) - len(kept), 'n_final': len(kept)})

print('Sample sizes (Table C1)')
print(pd.DataFrame(sample_sizes).to_string(index=False))

near_freezing_stats = {}
for variable in VARIABLES:
    records = []
    for reanalysis, df_all in pairs[variable].items():
        for station in STATIONS:
            df = df_all[df_all['station'] == station]
            # Temperature is evaluated on the LWS-only selection so the 5 °C tolerance does not bias the comparison
            key = (station, 'LWS') if variable == 'temperature' else (station, reanalysis)
            subset = df[df['time'].isin(near_freezing_times[key])]
            records.append({'station': station, 'reanalysis': reanalysis,
                            **comparison_stats(subset['obs'], subset['model'])})
    near_freezing_stats[variable] = pd.DataFrame.from_records(records)
    print(f'\n{variable.upper()} - near-freezing periods')
    print(near_freezing_stats[variable][COLUMNS_SHOWN].round(2).to_string(index=False))

## Precipitation phase observed at Sachs Harbour

In [ ]:
def classify_observed_phase(weather):
    """Phase of the observed weather conditions: 'Liquid' (drizzle, rain, freezing rain), 'Solid' (snow, ice pellets,
    hail), 'Mixed', 'None', or NaN when no weather was reported."""
    phases = []
    for report in weather:
        text = str(report).lower()
        liquid = 'drizzle' in text or 'rain' in text.replace('grain', '')
        solid = 'snow' in text or 'pellet' in text or 'hail' in text
        if liquid and solid:
            phases.append('Mixed')
        elif solid:
            phases.append('Solid')
        elif liquid:
            phases.append('Liquid')
        elif text == 'nan':
            phases.append(np.nan)
        else:
            phases.append('None')
    return np.array(phases, dtype=object)


def classify_model_phase(prern, presno, threshold):
    """Phase of the modelled hourly precipitation (mm/h): 'None' below `threshold`, otherwise the phase carrying
    at least half of the precipitation ('Mixed' if neither does)."""
    prern, presno = np.asarray(prern, dtype=float), np.asarray(presno, dtype=float)
    total = prern + presno
    phase = np.full(prern.shape, 'Mixed', dtype=object)
    none = (prern <= threshold) & (presno <= threshold)
    liquid = ~none & (prern >= total * 0.5)
    solid = ~none & ~liquid & (presno > total * 0.5)
    phase[none], phase[liquid], phase[solid] = 'None', 'Liquid', 'Solid'
    return phase


sachs = lws['SACHS_A'].dropna(subset=['weather']).copy()
sachs['phase'] = classify_observed_phase(sachs['weather'])

n_reports = len(sachs)
n_mixed = int((sachs['phase'] == 'Mixed').sum())
n_precip = int(sachs['phase'].isin(['Solid', 'Liquid', 'Mixed']).sum())
print(f'Sachs Harbour weather reports: {n_reports}')
print(f'    with precipitation: {n_precip}, of which mixed phase: {n_mixed} '
      f'({100 * n_mixed / n_reports:.2f}% of all reports, {100 * n_mixed / n_precip:.2f}% of precipitation reports)')

## Phase detection skill of each reanalysis-PPM pair (Fig. 5, Table 5)

In [ ]:
def align_on_common_times(near_freezing_time, model_time, obs_time):
    """Positions in the model and observation series of the timestamps present in all three inputs.
    Both position arrays come from one merge, so they always have the same length and pair the same timestamps."""
    common = set(near_freezing_time) & set(model_time) & set(obs_time)
    model_pos = pd.DataFrame({'time': model_time}).reset_index().rename(columns={'index': 'model_pos'})
    obs_pos = pd.DataFrame({'time': list(obs_time)}).reset_index().rename(columns={'index': 'obs_pos'})
    merged = pd.merge(model_pos[model_pos['time'].isin(common)],
                      obs_pos[obs_pos['time'].isin(common)], on='time').sort_values('time')
    return merged['model_pos'].to_numpy(), merged['obs_pos'].to_numpy()


def phase_scores(obs_phase, model_phase):
    """FBI, ETS and PC for solid and for liquid precipitation. Only hours where observation and model both report
    unambiguous solid or liquid precipitation are scored (mixed phase and dry hours are excluded)."""
    scored = np.isin(obs_phase, ['Solid', 'Liquid']) & np.isin(model_phase, ['Solid', 'Liquid'])
    obs, model = obs_phase[scored], model_phase[scored]
    rows = []
    for phase in ['Solid', 'Liquid']:
        a = int(((model == phase) & (obs == phase)).sum())
        b = int(((model == phase) & (obs != phase)).sum())
        c = int(((model != phase) & (obs == phase)).sum())
        d = int(((model != phase) & (obs != phase)).sum())
        rows.append({'phase': phase, **occurrence_scores(a, b, c, d)})
    return rows


# CaSR32_prediction shares its air temperature with CaSR32_analysis, so the analysis near-freezing times are used for both
TEMPERATURE_REFERENCE = {'CaSR32_analysis': 'CaSR32_analysis', 'CaSR32_prediction': 'CaSR32_analysis', 'ERA5': 'ERA5'}
hourly_threshold = TRACE_PRECIP_MM / 24

score_rows = []
for reanalysis in REANALYSES:
    times = near_freezing_times[('SACHS_A', TEMPERATURE_REFERENCE[reanalysis])]
    for ppm in PPMS:
        if (reanalysis, ppm) == ('ERA5', 'B00'):
            continue
        ds = ppm_ds.sel(forecast=reanalysis, PPM=ppm, location='SACHS_A')
        model_pos, obs_pos = align_on_common_times(times, pd.to_datetime(ds.time.values), sachs['time'])
        model_phase = classify_model_phase(ds['PRERN'].values[model_pos] * 3600,     # kg m-2 s-1 -> mm/h
                                           ds['PRESNO'].values[model_pos] * 3600, hourly_threshold)
        for row in phase_scores(sachs['phase'].to_numpy()[obs_pos], model_phase):
            score_rows.append({'reanalysis': reanalysis, 'ppm': ppm, **row})

phase_skill = pd.DataFrame(score_rows)
phase_skill.to_csv(RESULTS_DIR / 'ppm_phase_scores.csv', index=False)
print(f'Saved {RESULTS_DIR / "ppm_phase_scores.csv"} (read by Final_figures.ipynb for Fig. 5)\n')

solid = phase_skill[phase_skill['phase'] == 'Solid'].set_index(['reanalysis', 'ppm'])
print('Proportion correct, PC (Table 5; identical for solid and liquid)')
print(solid['PC'].unstack('ppm').reindex(columns=['J18', 'HP13', 'W19', 'B00']).round(2))
print('\nETS (Fig. 5, bottom row; identical for solid and liquid)')
print(solid['ETS'].unstack('ppm').reindex(columns=['J18', 'HP13', 'W19', 'B00']).round(2))
print('\nFBI (Fig. 5, top row)')
print(phase_skill.pivot_table(index=['reanalysis', 'phase'], columns='ppm', values='FBI')[['J18', 'HP13', 'W19', 'B00']].round(2))

## B00 freezing rain versus the other PPMs and the observations

B00 distinguishes freezing rain from rain. The other PPMs only return a liquid fraction, so the two are compared hour by hour.
B00's `PRERN`/`PRESNO` in the netCDF already merge freezing rain into liquid, so the four-phase output is re-read from the raw extraction.

In [ ]:
def pct(part, whole):
    return 100 * part / whole if whole else float('nan')


def b00_phase_series(product, station, threshold):
    """Dominant B00 phase per hour (Rain, Snow, Ice Pellets, Freezing Rain, or None below `threshold`, in mm/h)."""
    raw = read_b00_precip(RAW_MET_DIR / f'CaSR32_{product}' / f'extra_pcpn_casr3_{station}_8010_casr32_forcing.met') * 3600
    phase = raw.idxmax(axis=1)
    phase[raw.sum(axis=1) <= threshold] = 'None'
    return phase


def report_freezing_rain(scope, freezing_times, reanalysis, station, observed_freezing):
    """Share of B00 freezing-rain hours that the other PPMs call solid, and that were observed as freezing."""
    print(f'  [{scope}] {len(freezing_times)} B00 freezing-rain hours')
    for ppm in ['HP13', 'W19', 'J18']:
        ds = ppm_ds.sel(forecast=reanalysis, PPM=ppm, location=station)
        other = pd.Series(classify_model_phase(ds['PRERN'].values * 3600, ds['PRESNO'].values * 3600, hourly_threshold),
                          index=pd.to_datetime(ds.time.values))
        other = other.reindex(freezing_times).dropna()
        n_solid = int((other == 'Solid').sum())
        print(f'    {ppm} calls these hours solid in {n_solid}/{len(other)} cases ({pct(n_solid, len(other)):.1f}%)')

    with_obs = observed_freezing.reindex(freezing_times.intersection(observed_freezing.index))
    print(f'    Also observed as freezing: {int(with_obs.sum())}/{len(with_obs)} hours with a weather report '
          f'({pct(int(with_obs.sum()), len(with_obs)):.1f}%)')


observed_freezing = sachs['weather'].str.lower().str.contains('freezing', na=False)
observed_freezing.index = pd.to_datetime(sachs['time'].values)

for product in ['analysis', 'prediction']:
    reanalysis = f'CaSR32_{product}'
    print(f'\n=== {reanalysis} ===')
    b00 = b00_phase_series(product, 'SACHS_A', hourly_threshold)

    n_freezing = int((b00 == 'Freezing Rain').sum())
    print(f'  B00 predicts freezing rain in {n_freezing}/{len(b00)} hours ({pct(n_freezing, len(b00)):.2f}%)')
    report_freezing_rain('whole dataset', b00.index[b00 == 'Freezing Rain'], reanalysis, 'SACHS_A', observed_freezing)

    times = set(near_freezing_times[('SACHS_A', TEMPERATURE_REFERENCE[reanalysis])])
    b00_nf = b00[b00.index.isin(times)]
    n_freezing_nf = int((b00_nf == 'Freezing Rain').sum())
    n_liquid_nf = n_freezing_nf + int((b00_nf == 'Rain').sum())
    print(f'  Near-freezing: freezing rain in {n_freezing_nf}/{len(b00_nf)} hours ({pct(n_freezing_nf, len(b00_nf)):.2f}%), '
          f'i.e. {pct(n_freezing_nf, n_liquid_nf):.1f}% of its liquid hours ({n_freezing_nf}/{n_liquid_nf})')
    report_freezing_rain('near-freezing', b00_nf.index[b00_nf == 'Freezing Rain'], reanalysis, 'SACHS_A', observed_freezing)

# 6. Forcing of the sensitivity simulations (Section 2.5)

Both start from the CaSRv3.2-R + B00 forcing at Thomsen River and modify it between 19 September and 12 October 2003 (the ROS event):

* **no-rain**: all liquid precipitation is turned into snowfall (the warm temperatures are kept),
* **cold**: air temperature is capped at -0.5 °C (the precipitation phase, including freezing rain, is kept).

The output files are named `*_norain.met` and `*_cold.met` to match the manuscript.

In [ ]:
ROS_START = pd.Timestamp('2003-09-19')
ROS_END = pd.Timestamp('2003-10-12')
COLD_TEMPERATURE_CAP = -0.5   # deg C

source_file = PPM_DIR / 'B00' / 'basin_forcing_THOMSEN_RIVER_CaSR32_analysis.met'
forcing = pd.read_csv(source_file, sep=r'\s+', header=None, names=FORCING_COLUMNS).drop_duplicates()
forcing.index = met_timestamps(forcing)
in_event = (forcing.index >= ROS_START) & (forcing.index <= ROS_END)

# no-rain: liquid precipitation -> snow
rain_mm = forcing.loc[in_event, 'PRERN'] * 3600
print(f'Rain removed in the no-rain simulation: {rain_mm.sum():.2f} mm (maximum {rain_mm.max():.2f} mm/h)')
no_rain = forcing.copy()
no_rain.loc[in_event, 'PRERN'] = 0.0
no_rain.loc[in_event, 'PRESNO'] = no_rain.loc[in_event, 'PRE']
write_met_file(no_rain, SCENARIO_DIR / 'basin_forcing_THOMSEN_RIVER_CaSR32_analysis_norain.met')

# cold: air temperature capped
cold = forcing.copy()
n_capped = int((cold.loc[in_event, 'TA'] > COLD_TEMPERATURE_CAP).sum())
cold.loc[in_event, 'TA'] = cold.loc[in_event, 'TA'].clip(upper=COLD_TEMPERATURE_CAP)
print(f'Hours capped at {COLD_TEMPERATURE_CAP} °C in the cold simulation: {n_capped}')
write_met_file(cold, SCENARIO_DIR / 'basin_forcing_THOMSEN_RIVER_CaSR32_analysis_cold.met')

# 7. SVS2-Crocus ensemble results (Section 3.3.1)

Reads the locked-pasture summary of each 108-member ensemble (normal, cold, no-rain), written by the SVS2-Crocus post-processing.
Expected columns: `HOLD` (water holding capacity), `COMP` (snow compaction), `FALL` (falling snow density),
`DRIFT` (wind packing) and `locked_pasture` (True if the bottom 20 mm of the snowpack exceeds 350 kg m-3 on 31 October 2003).

In [ ]:
ENSEMBLE_SCENARIOS = ['ensemble', 'cold', 'norain']    # 'ensemble' = normal (unmodified forcing) simulation
PARAM_LABELS = {'HOLD': 'Water holding capacity', 'COMP': 'Snow compaction',
                'FALL': 'Falling snow density', 'DRIFT': 'Wind packing'}


def read_ensemble_summary(scenario):
    df = pd.read_csv(RESULTS_DIR / f'{scenario}_lockedpasture_summary_2003-10-31.csv', sep=None, engine='python')
    df['locked_pasture'] = df['locked_pasture'].astype(str).str.lower().eq('true')
    return df


def locked_pasture_share_by_scheme(df):
    """Number and share of members that simulated a locked pasture, for every option of every scheme."""
    tables = []
    for column, label in PARAM_LABELS.items():
        table = df.groupby(column)['locked_pasture'].agg(n_runs='size', n_locked='sum')
        table['pct_locked'] = (100 * table['n_locked'] / table['n_runs']).round(1)
        tables.append(table.reset_index().rename(columns={column: 'option'}).assign(scheme=label))
    return pd.concat(tables)[['scheme', 'option', 'n_runs', 'n_locked', 'pct_locked']].reset_index(drop=True)


ensembles = {scenario: read_ensemble_summary(scenario) for scenario in ENSEMBLE_SCENARIOS}

for scenario, df in ensembles.items():
    n_locked = int(df['locked_pasture'].sum())
    print(f'\n{scenario.upper()}: {n_locked} of {len(df)} members simulate a locked pasture ({100 * n_locked / len(df):.1f}%)')
    print(locked_pasture_share_by_scheme(df).to_string(index=False))

## Combinations of schemes that fail to form a locked pasture

In [ ]:
def combination_failure_rates(df, size):
    """For every combination of `size` schemes: members sharing those options that did NOT form a locked pasture."""
    rows = []
    for schemes in combinations(PARAM_LABELS, size):
        table = df.groupby(list(schemes))['locked_pasture'].agg(n_runs='size', n_failed=lambda s: int((~s).sum()))
        for options, row in table.iterrows():
            options = options if isinstance(options, tuple) else (options,)
            rows.append({'combination': ', '.join(f'{s}={o}' for s, o in zip(schemes, options)),
                         'n_failed': row['n_failed'], 'n_runs': row['n_runs'],
                         'pct_failed': round(100 * row['n_failed'] / row['n_runs'], 1)})
    return pd.DataFrame(rows).sort_values('pct_failed', ascending=False).reset_index(drop=True)


TOP_N = 10
for scenario, df in ensembles.items():
    for size in (2, 3):
        rates = combination_failure_rates(df, size)
        print(f'\n{scenario.upper()} - {size}-scheme combinations, {TOP_N} highest failure rates '
              f'({(rates["pct_failed"] == 100).sum()} of {len(rates)} combinations fail in every member)')
        print(rates.head(TOP_N).to_string(index=False))